# 04. 파생변수 기반 LightGBM 고도화

기존 `model_table_final.csv`에 2017-01-31 이전 정보만을 이용한
활동 변화량, 결제 패턴 및 만료 정합성 파생변수를 추가한다.

기존 Optuna 최적 파라미터와 동일한 조건으로 LightGBM을 학습하고,
기존 Valid PR-AUC 0.57362와 비교한다.

Test 데이터는 파생변수 채택 여부가 결정될 때까지 사용하지 않는다.

In [1]:
import json

from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
)


CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [
            CURRENT_DIR,
            *CURRENT_DIR.parents,
        ]
        if (
            (path / "preprocessing").is_dir()
            and (path / "modeling").is_dir()
            and (path / ".gitignore").exists()
        )
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "프로젝트 루트를 찾을 수 없습니다."
    )

PROCESSED_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

MODELS_DIR = PROJECT_ROOT / "models"

DATA_PATH = (
    PROCESSED_DIR
    / "model_table_final.csv"
)

PARAMS_PATH = (
    MODELS_DIR
    / "best_params.json"
)

ENHANCED_DATA_PATH = (
    PROCESSED_DIR
    / "model_table_enhanced.csv"
)

BASELINE_VALID_AUC = 0.8985848562
BASELINE_VALID_PR_AUC = 0.5736220968

CATEGORICAL_COLS = [
    "city",
    "registered_via",
    "gender",
]

DROP_COLS = [
    "msno",
    "snapshot",
    "split",
    "is_churn",
]

print("프로젝트 루트:", PROJECT_ROOT)
print("입력 데이터:", DATA_PATH)

프로젝트 루트: C:\Users\playdata2\Desktop\SKN34-2nd-2team
입력 데이터: C:\Users\playdata2\Desktop\SKN34-2nd-2team\data\processed\model_table_final.csv


## 파생변수 생성

기존 변수의 단순 크기뿐 아니라 최근 활동의 변화 방향,
거래당 결제액, 할인 정도, 결제일과 만료일의 정합성을 표현한다.

모든 변수는 관측 종료일인 2017-01-31 이전 데이터로 만들어진
기존 피처만 사용하므로 미래 정보는 포함하지 않는다.

In [2]:
df = pd.read_csv(
    DATA_PATH,
    low_memory=False,
)

assert df.shape == (992931, 44)
assert df["snapshot"].eq(
    "2017-01-31"
).all()
assert df.isna().sum().sum() == 0
assert df["msno"].duplicated().sum() == 0


# 기간별 실제 활동률
df["active_rate_7"] = (
    df["d7_active_days"] / 7
)

df["active_rate_30"] = (
    df["d30_active_days"] / 30
)

df["active_rate_90"] = (
    df["d90_active_days"] / 90
)


# 최근 활동량의 증가 또는 감소
df["activity_change_7_30"] = (
    df["active_rate_7"]
    - df["active_rate_30"]
)

df["activity_change_30_90"] = (
    df["active_rate_30"]
    - df["active_rate_90"]
)


# 최근 7일과 30일의 청취 행동 비교
df["secs_ratio_7_30"] = (
    df["d7_avg_total_secs"]
    / (df["d30_avg_total_secs"] + 1)
)

df["unq_ratio_7_30"] = (
    df["d7_avg_num_unq"]
    / (df["d30_avg_num_unq"] + 1)
)

df["completion_change_7_30"] = (
    df["d7_completion_rate"]
    - df["d30_completion_rate"]
)


# 거래당 평균 결제액
df["avg_amount_per_txn"] = (
    df["total_amount_paid"]
    / df["txn_count"].clip(lower=1)
)


# 마지막 결제의 할인율
df["last_discount_rate"] = (
    df["last_plan_list_price"]
    - df["last_actual_amount_paid"]
) / df[
    "last_plan_list_price"
].clip(lower=1)


# 마지막 결제일 + 요금제 기간과
# 실제 만료일까지의 차이
df["expiry_alignment_gap"] = (
    df["days_to_expire"]
    + df["days_since_last_txn"]
    - df["last_payment_plan_days"]
)

df["expiry_alignment_abs_gap"] = (
    df["expiry_alignment_gap"].abs()
)


# 마지막 요금제 기간 대비 남은 기간
df["remaining_plan_ratio"] = (
    df["days_to_expire"]
    / df[
        "last_payment_plan_days"
    ].clip(lower=1)
)


# 가입기간 대비 거래 빈도
df["txn_per_tenure_month"] = (
    df["txn_count"]
    / (
        df["tenure_days"] / 30
        + 1
    )
)


# 과거 자동결제 비율과 마지막 거래의 차이
df["auto_renew_change"] = (
    df["last_is_auto_renew"]
    - df["auto_renew_rate"]
)


# 자동결제 상태와 취소 이력의 상호작용
df["auto_cancel_interaction"] = (
    df["last_is_auto_renew"]
    * df["cancel_rate"]
)


DERIVED_COLS = [
    "active_rate_7",
    "active_rate_30",
    "active_rate_90",
    "activity_change_7_30",
    "activity_change_30_90",
    "secs_ratio_7_30",
    "unq_ratio_7_30",
    "completion_change_7_30",
    "avg_amount_per_txn",
    "last_discount_rate",
    "expiry_alignment_gap",
    "expiry_alignment_abs_gap",
    "remaining_plan_ratio",
    "txn_per_tenure_month",
    "auto_renew_change",
    "auto_cancel_interaction",
]

assert df[DERIVED_COLS].isna().sum().sum() == 0

assert np.isfinite(
    df[DERIVED_COLS].to_numpy()
).all()

print("기존 전체 컬럼 수: 44")
print("추가 파생변수 수:", len(DERIVED_COLS))
print("변경 후 전체 컬럼 수:", df.shape[1])
print("\n추가된 파생변수:")
print(DERIVED_COLS)

기존 전체 컬럼 수: 44
추가 파생변수 수: 16
변경 후 전체 컬럼 수: 60

추가된 파생변수:
['active_rate_7', 'active_rate_30', 'active_rate_90', 'activity_change_7_30', 'activity_change_30_90', 'secs_ratio_7_30', 'unq_ratio_7_30', 'completion_change_7_30', 'avg_amount_per_txn', 'last_discount_rate', 'expiry_alignment_gap', 'expiry_alignment_abs_gap', 'remaining_plan_ratio', 'txn_per_tenure_month', 'auto_renew_change', 'auto_cancel_interaction']


## Train/Validation 구성

기존 `model_table_final.csv`의 split을 그대로 사용한다.
범주형 변수의 category 정의는 전체 테이블에서 동일하게 적용하되,
모델 학습에는 Train 데이터만 사용한다.

Test 데이터는 아직 생성하거나 평가하지 않는다.

In [3]:
for column in CATEGORICAL_COLS:
    df[column] = df[column].astype(
        "category"
    )

FEATURE_COLS = [
    column
    for column in df.columns
    if column not in DROP_COLS
]

train_df = df[
    df["split"] == "train"
]

valid_df = df[
    df["split"] == "valid"
]

X_train = train_df[FEATURE_COLS]
y_train = train_df["is_churn"]

X_valid = valid_df[FEATURE_COLS]
y_valid = valid_df["is_churn"]

assert len(FEATURE_COLS) == 56
assert len(X_train) == 695051
assert len(X_valid) == 148940

train_set = lgb.Dataset(
    X_train,
    label=y_train,
    categorical_feature=CATEGORICAL_COLS,
)

valid_set = lgb.Dataset(
    X_valid,
    label=y_valid,
    categorical_feature=CATEGORICAL_COLS,
    reference=train_set,
)

print("전체 입력 피처:", len(FEATURE_COLS))
print("기존 피처:", 40)
print("추가 피처:", len(DERIVED_COLS))
print("Train:", X_train.shape)
print("Valid:", X_valid.shape)

전체 입력 피처: 56
기존 피처: 40
추가 피처: 16
Train: (695051, 56)
Valid: (148940, 56)


## 동일 파라미터 LightGBM 학습

파생변수 효과만 비교하기 위해 기존 Optuna 최적 파라미터를 그대로 사용한다.
Validation AUC를 기준으로 Early Stopping을 적용한 뒤 PR-AUC를 비교한다.

In [4]:
with open(
    PARAMS_PATH,
    "r",
    encoding="utf-8",
) as file:
    saved_config = json.load(file)

enhanced_params = {
    "objective": "binary",
    "metric": "auc",
    "seed": 42,
    "verbose": -1,
    "feature_pre_filter": False,
    **saved_config["params"],
}

enhanced_model = lgb.train(
    enhanced_params,
    train_set,
    num_boost_round=3000,
    valid_sets=[valid_set],
    valid_names=["valid"],
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
        ),
        lgb.log_evaluation(period=100),
    ],
)

valid_probabilities = enhanced_model.predict(
    X_valid,
    num_iteration=enhanced_model.best_iteration,
)

enhanced_valid_auc = roc_auc_score(
    y_valid,
    valid_probabilities,
)

enhanced_valid_pr_auc = (
    average_precision_score(
        y_valid,
        valid_probabilities,
    )
)

enhanced_valid_logloss = log_loss(
    y_valid,
    valid_probabilities,
)

auc_change = (
    enhanced_valid_auc
    - BASELINE_VALID_AUC
)

pr_auc_change = (
    enhanced_valid_pr_auc
    - BASELINE_VALID_PR_AUC
)

print("\n===== 파생변수 LightGBM VALID =====")
print(
    "Best iteration:",
    enhanced_model.best_iteration,
)
print(
    "AUC:",
    f"{enhanced_valid_auc:.6f}",
)
print(
    "PR-AUC:",
    f"{enhanced_valid_pr_auc:.6f}",
)
print(
    "LogLoss:",
    f"{enhanced_valid_logloss:.6f}",
)

print("\n===== 기존 모델 대비 =====")
print(
    "AUC 변화:",
    f"{auc_change:+.6f}",
)
print(
    "PR-AUC 변화:",
    f"{pr_auc_change:+.6f}",
)

if pr_auc_change > 0:
    print(
        "\n파생변수 모델의 Valid PR-AUC가 개선됐습니다."
    )
else:
    print(
        "\n파생변수 모델의 Valid PR-AUC가 개선되지 않았습니다."
    )

Training until validation scores don't improve for 100 rounds
[100]	valid's auc: 0.899784
[200]	valid's auc: 0.900757
[300]	valid's auc: 0.900862
[400]	valid's auc: 0.900706
Early stopping, best iteration is:
[325]	valid's auc: 0.900936
Evaluated only: auc

===== 파생변수 LightGBM VALID =====
Best iteration: 325
AUC: 0.900936
PR-AUC: 0.582655
LogLoss: 0.143800

===== 기존 모델 대비 =====
AUC 변화: +0.002351
PR-AUC 변화: +0.009033

파생변수 모델의 Valid PR-AUC가 개선됐습니다.


In [5]:
# 5. Validation에서 F1 기준 최적 임계값 결정

from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

precisions, recalls, thresholds = (
    precision_recall_curve(
        y_valid,
        valid_probabilities,
    )
)

f1_scores = (
    2 * precisions[:-1] * recalls[:-1]
    / (
        precisions[:-1]
        + recalls[:-1]
        + 1e-12
    )
)

best_index = np.argmax(f1_scores)

enhanced_threshold = float(
    thresholds[best_index]
)

valid_predictions = (
    valid_probabilities
    >= enhanced_threshold
).astype(int)

print(
    "최적 Threshold:",
    f"{enhanced_threshold:.6f}",
)
print(
    "Valid Precision:",
    f"{precision_score(y_valid, valid_predictions):.6f}",
)
print(
    "Valid Recall:",
    f"{recall_score(y_valid, valid_predictions):.6f}",
)
print(
    "Valid F1:",
    f"{f1_score(y_valid, valid_predictions):.6f}",
)

최적 Threshold: 0.262053
Valid Precision: 0.532779
Valid Recall: 0.578721
Valid F1: 0.554800


In [6]:
# 6. 확정된 모델과 임계값으로 Test 최종 평가

test_df = df[
    df["split"] == "test"
]

X_test = test_df[FEATURE_COLS]
y_test = test_df["is_churn"]

assert len(X_test) == 148940

test_probabilities = enhanced_model.predict(
    X_test,
    num_iteration=enhanced_model.best_iteration,
)

test_predictions = (
    test_probabilities
    >= enhanced_threshold
).astype(int)

enhanced_test_auc = roc_auc_score(
    y_test,
    test_probabilities,
)

enhanced_test_pr_auc = (
    average_precision_score(
        y_test,
        test_probabilities,
    )
)

enhanced_test_logloss = log_loss(
    y_test,
    test_probabilities,
)

enhanced_test_precision = precision_score(
    y_test,
    test_predictions,
)

enhanced_test_recall = recall_score(
    y_test,
    test_predictions,
)

enhanced_test_f1 = f1_score(
    y_test,
    test_predictions,
)

enhanced_test_cm = confusion_matrix(
    y_test,
    test_predictions,
)

print("===== 파생변수 LightGBM TEST =====")
print(
    "AUC:",
    f"{enhanced_test_auc:.6f}",
)
print(
    "PR-AUC:",
    f"{enhanced_test_pr_auc:.6f}",
)
print(
    "LogLoss:",
    f"{enhanced_test_logloss:.6f}",
)
print(
    "Precision:",
    f"{enhanced_test_precision:.6f}",
)
print(
    "Recall:",
    f"{enhanced_test_recall:.6f}",
)
print(
    "F1:",
    f"{enhanced_test_f1:.6f}",
)
print("\nConfusion Matrix:")
print(enhanced_test_cm)

print("\n===== 기존 LightGBM 대비 =====")
print(
    "Test AUC 변화:",
    f"{enhanced_test_auc - 0.9011266853:+.6f}",
)
print(
    "Test PR-AUC 변화:",
    f"{enhanced_test_pr_auc - 0.5752987190:+.6f}",
)

===== 파생변수 LightGBM TEST =====
AUC: 0.903453
PR-AUC: 0.582491
LogLoss: 0.143138
Precision: 0.529445
Recall: 0.582668
F1: 0.554783

Confusion Matrix:
[[134490   4930]
 [  3973   5547]]

===== 기존 LightGBM 대비 =====
Test AUC 변화: +0.002326
Test PR-AUC 변화: +0.007192


In [7]:
# 5. PR-AUC 기준으로 Early Stopping하여 재학습

pr_params = enhanced_params.copy()

pr_params["metric"] = "average_precision"

pr_model = lgb.train(
    pr_params,
    train_set,
    num_boost_round=3000,
    valid_sets=[valid_set],
    valid_names=["valid"],
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
        ),
        lgb.log_evaluation(period=100),
    ],
)

pr_valid_probabilities = pr_model.predict(
    X_valid,
    num_iteration=pr_model.best_iteration,
)

pr_valid_auc = roc_auc_score(
    y_valid,
    pr_valid_probabilities,
)

pr_valid_pr_auc = average_precision_score(
    y_valid,
    pr_valid_probabilities,
)

pr_valid_logloss = log_loss(
    y_valid,
    pr_valid_probabilities,
)

print("\n===== PR-AUC 기준 Early Stopping =====")
print(
    "Best iteration:",
    pr_model.best_iteration,
)
print(
    "Valid AUC:",
    f"{pr_valid_auc:.6f}",
)
print(
    "Valid PR-AUC:",
    f"{pr_valid_pr_auc:.6f}",
)
print(
    "Valid LogLoss:",
    f"{pr_valid_logloss:.6f}",
)

print("\n===== AUC 기준 모델 대비 =====")
print(
    "PR-AUC 변화:",
    f"{pr_valid_pr_auc - enhanced_valid_pr_auc:+.6f}",
)

Training until validation scores don't improve for 100 rounds
[100]	valid's average_precision: 0.57944
[200]	valid's average_precision: 0.58198
[300]	valid's average_precision: 0.582531
[400]	valid's average_precision: 0.582454
Early stopping, best iteration is:
[347]	valid's average_precision: 0.582737
Evaluated only: average_precision

===== PR-AUC 기준 Early Stopping =====
Best iteration: 347
Valid AUC: 0.900852
Valid PR-AUC: 0.582737
Valid LogLoss: 0.143777

===== AUC 기준 모델 대비 =====
PR-AUC 변화: +0.000082


In [8]:
# 5. 파생변수 16개가 추가된 v1 테이블 저장
# 기존 model_table_final.csv는 덮어쓰지 않음

ENHANCED_V1_PATH = (
    PROCESSED_DIR
    / "model_table_enhanced_v1.csv"
)

df.to_csv(
    ENHANCED_V1_PATH,
    index=False,
)

file_size_mb = (
    ENHANCED_V1_PATH.stat().st_size
    / 1024**2
)

print("저장 경로:", ENHANCED_V1_PATH)
print("shape:", df.shape)
print("파일 크기:", f"{file_size_mb:,.1f} MB")
print("Enhanced v1 저장 완료")

저장 경로: C:\Users\playdata2\Desktop\SKN34-2nd-2team\data\processed\model_table_enhanced_v1.csv
shape: (992931, 60)
파일 크기: 564.5 MB
Enhanced v1 저장 완료


In [9]:
# 7. PR-AUC 기준 모델을 enhanced v1 후보로 선택하고 저장

from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
)

enhanced_model = pr_model
valid_probabilities = pr_valid_probabilities

enhanced_valid_auc = pr_valid_auc
enhanced_valid_pr_auc = pr_valid_pr_auc
enhanced_valid_logloss = pr_valid_logloss


# Validation에서 F1 기준 임계값 결정
precisions, recalls, thresholds = (
    precision_recall_curve(
        y_valid,
        valid_probabilities,
    )
)

f1_scores = (
    2 * precisions[:-1] * recalls[:-1]
    / (
        precisions[:-1]
        + recalls[:-1]
        + 1e-12
    )
)

best_index = np.argmax(f1_scores)

enhanced_threshold = float(
    thresholds[best_index]
)

valid_predictions = (
    valid_probabilities
    >= enhanced_threshold
).astype(int)

enhanced_valid_precision = precision_score(
    y_valid,
    valid_predictions,
)

enhanced_valid_recall = recall_score(
    y_valid,
    valid_predictions,
)

enhanced_valid_f1 = f1_score(
    y_valid,
    valid_predictions,
)


# 모델 저장
V1_MODEL_PATH = (
    MODELS_DIR
    / "lightgbm_enhanced_v1.txt"
)

V1_META_PATH = (
    MODELS_DIR
    / "lightgbm_enhanced_v1_meta.json"
)

V1_IMPORTANCE_PATH = (
    MODELS_DIR
    / "lightgbm_enhanced_v1_importance.csv"
)

enhanced_model.save_model(
    str(V1_MODEL_PATH)
)


# 피처 중요도 저장
feature_importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "gain_importance": (
        enhanced_model.feature_importance(
            importance_type="gain"
        )
    ),
    "split_importance": (
        enhanced_model.feature_importance(
            importance_type="split"
        )
    ),
})

feature_importance = (
    feature_importance
    .sort_values(
        "gain_importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

feature_importance.to_csv(
    V1_IMPORTANCE_PATH,
    index=False,
)


# 모델 설정과 성능 저장
v1_metadata = {
    "model_name": (
        "LightGBM enhanced v1"
    ),
    "feature_cutoff": "2017-01-31",
    "early_stopping_metric": (
        "average_precision"
    ),
    "best_iteration": int(
        enhanced_model.best_iteration
    ),
    "threshold_source": (
        "Validation F1 maximum"
    ),
    "threshold": enhanced_threshold,
    "feature_count": len(FEATURE_COLS),
    "derived_feature_count": len(
        DERIVED_COLS
    ),
    "derived_features": DERIVED_COLS,
    "categorical_features": (
        CATEGORICAL_COLS
    ),
    "feature_cols": FEATURE_COLS,
    "params": enhanced_params,
    "valid_auc": float(
        enhanced_valid_auc
    ),
    "valid_pr_auc": float(
        enhanced_valid_pr_auc
    ),
    "valid_logloss": float(
        enhanced_valid_logloss
    ),
    "valid_precision": float(
        enhanced_valid_precision
    ),
    "valid_recall": float(
        enhanced_valid_recall
    ),
    "valid_f1": float(
        enhanced_valid_f1
    ),
    "baseline_valid_pr_auc": (
        BASELINE_VALID_PR_AUC
    ),
    "pr_auc_improvement": float(
        enhanced_valid_pr_auc
        - BASELINE_VALID_PR_AUC
    ),
}

with open(
    V1_META_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        v1_metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Threshold:", f"{enhanced_threshold:.6f}")
print("Valid Precision:", f"{enhanced_valid_precision:.6f}")
print("Valid Recall:", f"{enhanced_valid_recall:.6f}")
print("Valid F1:", f"{enhanced_valid_f1:.6f}")

print("\n모델:", V1_MODEL_PATH)
print("메타데이터:", V1_META_PATH)
print("중요도:", V1_IMPORTANCE_PATH)
print("\nEnhanced v1 후보 저장 완료")

Threshold: 0.268762
Valid Precision: 0.538219
Valid Recall: 0.573154
Valid F1: 0.555137

모델: C:\Users\playdata2\Desktop\SKN34-2nd-2team\models\lightgbm_enhanced_v1.txt
메타데이터: C:\Users\playdata2\Desktop\SKN34-2nd-2team\models\lightgbm_enhanced_v1_meta.json
중요도: C:\Users\playdata2\Desktop\SKN34-2nd-2team\models\lightgbm_enhanced_v1_importance.csv

Enhanced v1 후보 저장 완료
